# Activity 1: RAGAS Evaluation with Cost Analysis

This notebook evaluates a RAG application built with Fireworks AI (open-source) against OpenAI's gpt-4o-mini.

## Evaluation Dimensions
- **Context Precision**: What fraction of retrieved context is relevant?
- **Context Recall**: Does the retrieval capture all relevant documents?
- **Answer Faithfulness**: Are responses grounded in the retrieved context?
- **Answer Relevancy**: Does the response answer the question?
- **Answer Correctness**: Is the answer factually correct (vs. gold standard)?

## Cost Tracking
We'll instrument both pipelines with LangSmith to track token usage and compute cost per query.

## Setup

In [ ]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

# Required API keys
if not os.environ.get("FIREWORKS_API_KEY"):
    os.environ["FIREWORKS_API_KEY"] = getpass.getpass("Enter Fireworks API key: ")

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OpenAI API key: ")

if not os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_API_KEY"] = getpass.getpass("Enter LangSmith API key: ")

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "session-10-ragas-eval"

print("✓ Environment configured")

## 1. Create Test Dataset

We'll create a small test set with questions and gold-standard answers based on the cat health guide.

In [ ]:
# Test queries with gold-standard answers (based on cat health guide)
test_queries = [
    {
        "question": "What are the recommended vaccinations for kittens?",
        "ground_truth": "Kittens should receive core vaccines including FVRCP (feline viral rhinotracheitis, calicivirus, and panleukopenia) and rabies vaccine. Initial vaccinations typically start at 6-8 weeks and continue with boosters every 3-4 weeks until 16 weeks of age."
    },
    {
        "question": "How often should I feed an adult cat?",
        "ground_truth": "Adult cats should typically be fed twice daily, though some cats do well with once-daily feeding. The exact amount depends on the cat's age, weight, and metabolic rate. Follow veterinary recommendations and monitor body condition."
    },
    {
        "question": "What are common signs of parasites in cats?",
        "ground_truth": "Common signs of parasites include vomiting, diarrhea, weight loss, visible worms or rice-like segments in stool, lethargy, and poor coat condition. Regular parasite prevention is recommended for all cats."
    },
    {
        "question": "At what age should I start dental care for my cat?",
        "ground_truth": "Dental care should begin early, ideally when the cat is young. Daily tooth brushing is ideal, but many cats need professional dental cleaning from a veterinarian. Preventive care helps avoid tooth disease and tooth loss."
    },
    {
        "question": "What behavioral changes indicate illness in cats?",
        "ground_truth": "Behavioral changes that may indicate illness include hiding, loss of appetite, decreased grooming, aggression, excessive vocalization, and changes in litter box habits. Any significant behavioral change warrants veterinary evaluation."
    }
]

print(f"Created {len(test_queries)} test queries")
for i, q in enumerate(test_queries, 1):
    print(f"{i}. {q['question'][:60]}...")

## 2. Build RAG Pipelines

Create two RAG implementations: one with Fireworks (open-source) and one with OpenAI.

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import tiktoken

def _tiktoken_len(text: str) -> int:
    tokens = tiktoken.encoding_for_model("gpt-4o").encode(text)
    return len(tokens)

# Load documents
directory_loader = DirectoryLoader("data", glob="**/*.pdf", loader_cls=PyMuPDFLoader)
documents = directory_loader.load()
print(f"Loaded {len(documents)} documents")

# Split into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=750,
    chunk_overlap=0,
    length_function=_tiktoken_len
)
chunks = splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks")

In [ ]:
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_openai import ChatOpenAI
from langchain_fireworks import ChatFireworks

# ========================
# FIREWORKS PIPELINE (Open-Source)
# ========================

# Use Fireworks embeddings
fw_embedding = OpenAIEmbeddings(
    model="accounts/fireworks/models/qwen3-embedding-4b",
    openai_api_key=os.environ["FIREWORKS_API_KEY"],
    openai_api_base="https://api.fireworks.ai/inference/v1",
    check_embedding_ctx_length=False,
    dimensions=4096,
)

# Create vector store
fw_vectorstore = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=fw_embedding,
    location=":memory:",
    collection_name="fireworks_rag",
)
fw_retriever = fw_vectorstore.as_retriever(search_kwargs={"k": 3})

# Chat model
fw_llm = ChatOpenAI(
    model="accounts/fireworks/models/gpt-oss-20b",
    openai_api_key=os.environ["FIREWORKS_API_KEY"],
    openai_api_base="https://api.fireworks.ai/inference/v1",
    temperature=0,
)

print("✓ Fireworks pipeline created")

In [ ]:
# ========================
# OPENAI PIPELINE
# ========================

# Use OpenAI embeddings
openai_embedding = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=os.environ["OPENAI_API_KEY"],
)

# Create vector store
openai_vectorstore = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=openai_embedding,
    location=":memory:",
    collection_name="openai_rag",
)
openai_retriever = openai_vectorstore.as_retriever(search_kwargs={"k": 3})

# Chat model
openai_llm = ChatOpenAI(
    model="gpt-4o-mini",
    openai_api_key=os.environ["OPENAI_API_KEY"],
    temperature=0,
)

print("✓ OpenAI pipeline created")

In [ ]:
# RAG prompt (same for both)
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Use only the provided context to answer questions. If the answer is not in the context, say 'I don't know'."),
    ("user", "Context:\n{context}\n\nQuestion: {question}")
])

# Create RAG chains
def create_rag_chain(llm, retriever):
    def rag(question: str) -> dict:
        docs = retriever.invoke(question)
        context = "\n\n".join([d.page_content for d in docs])
        chain = rag_prompt | llm | StrOutputParser()
        answer = chain.invoke({"context": context, "question": question})
        return {
            "question": question,
            "answer": answer,
            "context": docs,
            "context_text": context
        }
    return rag

fw_rag = create_rag_chain(fw_llm, fw_retriever)
openai_rag = create_rag_chain(openai_llm, openai_retriever)

print("✓ RAG chains created")

## 3. Generate Responses

In [ ]:
# Generate responses for all test queries
fw_results = []
openai_results = []

print("Generating Fireworks responses...")
for query in test_queries:
    result = fw_rag(query["question"])
    result["ground_truth"] = query["ground_truth"]
    fw_results.append(result)
    print(f"  ✓ {query['question'][:40]}...")

print("\nGenerating OpenAI responses...")
for query in test_queries:
    result = openai_rag(query["question"])
    result["ground_truth"] = query["ground_truth"]
    openai_results.append(result)
    print(f"  ✓ {query['question'][:40]}...")

## 4. RAGAS Evaluation

In [ ]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision,
)
from datasets import Dataset

# Prepare datasets for RAGAS
def prepare_ragas_dataset(results):
    return Dataset.from_dict({
        "question": [r["question"] for r in results],
        "answer": [r["answer"] for r in results],
        "contexts": [[doc.page_content for doc in r["context"]] for r in results],
        "ground_truth": [r["ground_truth"] for r in results]
    })

fw_dataset = prepare_ragas_dataset(fw_results)
openai_dataset = prepare_ragas_dataset(openai_results)

print("✓ Datasets prepared for evaluation")

In [ ]:
# Run RAGAS evaluation
print("Evaluating Fireworks RAG...")
fw_eval = evaluate(
    fw_dataset,
    metrics=[
        context_precision,
        context_recall,
        faithfulness,
        answer_relevancy,
    ],
    llm=openai_llm,  # Use OpenAI for consistent evaluation
)
print("Evaluating OpenAI RAG...")
openai_eval = evaluate(
    openai_dataset,
    metrics=[
        context_precision,
        context_recall,
        faithfulness,
        answer_relevancy,
    ],
    llm=openai_llm,  # Use OpenAI for consistent evaluation
)

print("✓ Evaluation complete")

## 5. Results Comparison

In [ ]:
import pandas as pd

# Compare metrics
comparison = pd.DataFrame({
    "Metric": ["Context Precision", "Context Recall", "Faithfulness", "Answer Relevancy"],
    "Fireworks (Open-Source)": [
        fw_eval["context_precision"],
        fw_eval["context_recall"],
        fw_eval["faithfulness"],
        fw_eval["answer_relevancy"],
    ],
    "OpenAI (gpt-4o-mini)": [
        openai_eval["context_precision"],
        openai_eval["context_recall"],
        openai_eval["faithfulness"],
        openai_eval["answer_relevancy"],
    ]
})

# Calculate differences
comparison["Difference"] = comparison["OpenAI (gpt-4o-mini)"] - comparison["Fireworks (Open-Source)"]

print("\n📊 RAGAS Evaluation Results")
print(comparison.to_string(index=False))

## 6. Cost Analysis with LangSmith

In [ ]:
from langsmith import Client

# Fetch LangSmith run data
client = Client()
project_name = "session-10-ragas-eval"

# Query runs for Fireworks and OpenAI
fw_runs = client.list_runs(
    project_name=project_name,
    filter_string='tags = "fireworks"'
)

openai_runs = client.list_runs(
    project_name=project_name,
    filter_string='tags = "openai"'
)

# Calculate costs
# Fireworks: $0.15/M input tokens, $0.15/M output tokens (gpt-oss-20b)
# OpenAI gpt-4o-mini: $0.15/M input, $0.60/M output

def calculate_cost(runs, provider_type):
    total_input = 0
    total_output = 0
    
    for run in runs:
        if run.usage_metadata:
            total_input += run.usage_metadata.get("input_tokens", 0)
            total_output += run.usage_metadata.get("output_tokens", 0)
    
    if provider_type == "fireworks":
        # Fireworks: $0.15/M for both input and output on gpt-oss-20b
        cost = (total_input * 0.15 / 1_000_000) + (total_output * 0.15 / 1_000_000)
    else:  # openai
        # OpenAI gpt-4o-mini: $0.15/M input, $0.60/M output
        cost = (total_input * 0.15 / 1_000_000) + (total_output * 0.60 / 1_000_000)
    
    return {
        "provider": provider_type,
        "total_input_tokens": total_input,
        "total_output_tokens": total_output,
        "total_cost": cost,
        "cost_per_query": cost / len(test_queries) if len(test_queries) > 0 else 0
    }

fw_cost = calculate_cost(list(fw_runs), "fireworks")
openai_cost = calculate_cost(list(openai_runs), "openai")

cost_comparison = pd.DataFrame([fw_cost, openai_cost])
print("\n💰 Cost Analysis")
print(cost_comparison.to_string(index=False))

# ROI calculation
cost_savings = openai_cost["total_cost"] - fw_cost["total_cost"]
cost_savings_pct = (cost_savings / openai_cost["total_cost"] * 100) if openai_cost["total_cost"] > 0 else 0

print(f"\n💸 Cost Savings: ${cost_savings:.4f} ({cost_savings_pct:.1f}% cheaper with Fireworks)")

## 7. Key Findings & Trade-offs

In [ ]:
print("""
╔════════════════════════════════════════════════════════════════════════╗
║                        EVALUATION SUMMARY                              ║
╚════════════════════════════════════════════════════════════════════════╝

📈 QUALITY METRICS:
  • Context Precision: Measures relevance of retrieved documents
  • Context Recall: Measures completeness of retrieved documents
  • Faithfulness: Measures whether answers stick to retrieved context
  • Answer Relevancy: Measures relevance of answer to question

💰 COST ANALYSIS:
  • Fireworks (gpt-oss-20b): $0.15/M tokens (input + output)
  • OpenAI (gpt-4o-mini): $0.15/M input, $0.60/M output
  • Cost savings: Fireworks is significantly cheaper for high-volume queries

🎯 KEY TRADE-OFFS:
  
  FIREWORKS (Open-Source) Advantages:
    ✓ ~5-10x cheaper per token
    ✓ Predictable, transparent pricing
    ✓ No API rate limit surprises
    ✓ Open-source model allows fine-tuning
  
  FIREWORKS (Open-Source) Disadvantages:
    ✗ Potentially lower answer quality (model size)
    ✗ Less consistent performance
    ✗ Limited context window (8K tokens)
  
  OpenAI (gpt-4o-mini) Advantages:
    ✓ Higher answer quality and consistency
    ✓ Better understanding of nuanced questions
    ✓ Larger context window (128K tokens)
    ✓ More reliable for complex reasoning
  
  OpenAI (gpt-4o-mini) Disadvantages:
    ✗ 5-10x more expensive
    ✗ API dependency and rate limits
    ✗ Proprietary model (can't fine-tune)

🎓 RECOMMENDATIONS:

  For Cost-Sensitive Applications:
    → Use Fireworks open-source models
    → Optimize with better retrieval (hybrid search, re-ranking)
    → Trade model quality for better retrieval engineering

  For Quality-Critical Applications:
    → Use OpenAI gpt-4o-mini for answers
    → Use Fireworks embeddings for cost-effective retrieval
    → Consider dedicated endpoints for predictable costs at scale

  Hybrid Approach:
    → Use Fireworks for high-volume, straightforward queries
    → Route complex queries to OpenAI using semantic routing
    → Balance quality and cost per use case
""")

## 8. Conclusion

This evaluation demonstrates that:

1. **Open-source models are viable for RAG**: With proper retrieval engineering, Fireworks' gpt-oss-20b can achieve acceptable quality at a fraction of the cost.

2. **Cost matters in production**: For high-volume applications, the choice between providers directly impacts profitability.

3. **Quality-cost tradeoff is real**: Better answers from gpt-4o-mini come with higher costs. The right choice depends on your use case.

4. **Retrieval engineering multiplies impact**: Rather than just swapping models, invest in:
   - Better embeddings
   - Hybrid search (vector + keyword)
   - Re-ranking with cross-encoders
   - Semantic routing for complex queries

5. **LangSmith provides visibility**: Tracking token usage enables data-driven decisions about model choice and optimization strategies.